In [1]:
import sys
sys.path.append("..")   # so notebook can find the app/ folder

from app.core.rag_chain import build_rag_chain, create_memory, parse_response

# Create a fresh memory + chain
memory = create_memory()
chain  = build_rag_chain(memory)

print("Chain ready ✅")

c:\Users\My pc\OneDrive\Documents\Desktop\rag-chatbot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\My pc\OneDrive\Documents\Desktop\rag-chatbot\notebooks\..\app\core\rag_chain.py:68: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory(


🤖 Loading embedding model 'sentence-transformers/all-MiniLM-L6-v2'...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6174.26it/s]


✅ Embedding model loaded
✅ Retriever ready — will fetch top 4 chunks per query
✅ Retriever initialized (cached)
✅ LLM initialized (cached)
✅ RAG chain built successfully
Chain ready ✅


In [2]:
question = "What is the recommended treatment for hypertension?"

response = chain.invoke({"question": question})
result   = parse_response(response)

print("Answer:")
print(result["answer"])
print("\nSources:")
for src in result["sources"]:
    print(f"  📄 {src['file']} — page {src['page']}")



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
You are a helpful medical information assistant.
Answer the user's question using ONLY the context provided below.
If the answer is not in the context, say "I don't have enough information in my documents to answer this. Please consult a medical professional."

Context:
therapy especially in low resource settings are recommended with or without risk 
assessment.    
▪  Lifestyle modifications and use of anti-hypertensives medications are 
recommended for treatment either alone or in combination. 
▪ Lifestyle measures are recommended for all grades of hypertension.  
▪ Initial one monthly follow-up is recommended until target blood pressure is 
achieved and 3-6 monthly follow up afterwards.28,29 Deployment of tele monitoring 
of BP may enable longer follow up intervals in both cases.30 
▪ Non-physician health workers including nurses, pharmacists and community health 
workers who ha

In [4]:
# This follow-up question relies on the previous answer being in memory
followup = "What are the side effects of those medications?"

response2 = chain.invoke({"question": followup})
result2   = parse_response(response2)

print("Follow-up Answer:")
print(result2["answer"])
print("\nSources:")
for src in result2["sources"]:
    print(f"  📄 {src['file']} — page {src['page']}")



> Entering new LLMChain chain...
Prompt after formatting:
Given the following conversation and a follow up question, rephrase the follow up question to be a standalone question, in its original language.

Chat History:

Human: What is the recommended treatment for hypertension?
Assistant: Lifestyle modifications and the use of anti-hypertensive medications, either alone or in combination, are recommended for the treatment of hypertension. Lifestyle measures are recommended for all grades of hypertension and include regular physical activity, reduction of alcohol consumption, and reduction of salt consumption (<5g salt per day).

Five classes of drugs are broadly used for pharmacological treatment: Calcium channel blockers (CCBs), Beta blockers, Diuretics, Angiotensin converting enzyme inhibitors (ACEIs), and Angiotensin receptor blockers (ARBs).
Follow Up Input: What are the side effects of those medications?
Standalone question:

> Finished chain.


> Entering new StuffDocumentsChai

In [4]:
import sys
sys.path.append("..")

from app.core.agent import build_agent, extract_text_from_output

agent = build_agent()

response1 = agent.invoke({
    "input": "What is the maximum daily dose of paracetamol for adults?",
    "chat_history": []
})

print("Answer:", extract_text_from_output(response1["output"]))
print("\nTools used:", [step[0].tool for step in response1["intermediate_steps"]])

✅ Agent built with tools: search_medical_docs, search_web


> Entering new AgentExecutor chain...

Invoking: `search_medical_docs` with `{'query': 'maximum daily dose of paracetamol for adults'}`


[content_7.pdf, page 243]: Page 244 / 663
PARACETAMOL = ACETAMINOPHEN oral
Last updated: October 2024
 
Do not exceed indicated doses, especially in children and older patients. Paracetamol
intoxications are severe (hepatic cytolysis).
 
Therapeutic action
Indications
Forms and strengths
Dosage
 
 
Analgesic, antipyretic
Mild pain
Fever
100 mg and 500 mg tablets
100 mg dispersible tablet
120 mg/5 ml oral suspension
Child under 1 month: 10 mg/kg 3 or 4 times daily (max. 40 mg/kg daily) 
Child 1 month and over: 15 mg/kg 3 or 4 times daily (max. 60 mg/kg daily)
Adult: 1 g 3 or 4 times daily (max. 4 g daily)

[content_7.pdf, page 469]: Page 470 / 663
PARACETAMOL = ACETAMINOPHEN
injectable
Last updated: February 2024
 
Prescription under medical supervision
 
Do not exceed indicated doses, especi

In [6]:
# Test 2: Question that likely needs web search (not in WHO/AIIMS PDFs)
response2 = agent.invoke({
    "input": "What are the latest WHO health emergency declarations in 2026?",
    "chat_history": []
})
print("Answer:", extract_text_from_output(response2["output"]))
print("\nTools used:", [step[0].tool for step in response2["intermediate_steps"]])



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': 'latest WHO health emergency declarations 2026'}`


[{'title': 'List of Public Health Emergency Declarations - ASPR', 'url': 'https://aspr.hhs.gov/legal/PHE/pages/default.aspx', 'content': 'Hawaii June 8, 2026 Renewal of Determination that a Public Health Emergency Exists Nationwide. Opioid Crisis on March 17, 2026 Opioid Crisis', 'score': 0.7978881}, {'title': 'WHO declares new Ebola outbreak in Africa a global health emergency : NPR', 'url': 'https://www.npr.org/2026/05/18/nx-s1-5824998/who-declares-new-ebola-outbreak-in-africa-a-global-health-emergency', 'content': "FADEL: That's NPR's Jonathan Lambert. Thank you for your reporting, Jonathan.\n\nLAMBERT: Thank you.\n\nCopyright © 2026 NPR. All rights reserved. Visit our website terms of use and permissions pages at www.npr.org for further information.\n\nAccuracy and availability of NPR transcripts may vary. Transcript text may be revised